In [16]:
import os
import json
from dotenv import load_dotenv

load_dotenv()

checks = {
    "GROQ_API_KEY": os.getenv("GROQ_API_KEY"),
    "HUGGINGFACE_API_TOKEN": os.getenv("HUGGINGFACE_API_TOKEN"),
    "ASTRA_DB_APPLICATION_TOKEN": os.getenv("ASTRA_DB_APPLICATION_TOKEN"),
    "ASTRA_DB_API_ENDPOINT": os.getenv("ASTRA_DB_API_ENDPOINT"),
    "DATABASE_URL": os.getenv("DATABASE_URL"),
}

#### Agent State


In [17]:
from typing import TypedDict, Literal, Optional
from langchain_core.messages import BaseMessage


class AgentState(TypedDict):
    user_id: str
    query: str
    image: Optional[str]

    # Route
    intent: Optional[Literal["ingest", "rag_query", "support_query"]]
    confidence: Optional[float]

    # RAG
    chunks: Optional[list[str]]
    sources: Optional[list[dict]]
    retrieved_docs: Optional[list[str]]

    # Support
    ticket_category: Optional[str]
    ticket_id: Optional[str]
    escalate: Optional[bool]

    # Output
    messages: list[BaseMessage]
    final_response: Optional[str]


print("✅ AgentState defined")
print("Fields:", list(AgentState.__annotations__.keys()))

✅ AgentState defined
Fields: ['user_id', 'query', 'image', 'intent', 'confidence', 'chunks', 'sources', 'retrieved_docs', 'ticket_category', 'ticket_id', 'escalate', 'messages', 'final_response']


#### Intent Node


In [18]:
from groq import Groq

client = Groq(api_key=os.getenv("GROQ_API_KEY"))

INTENT_SYSTEM_PROMPT = """
You are an intent classifier for an AI assistant called SageDesk.
Your job is to classify the user's query into exactly one of three intents:

1. "ingest"        — user wants to upload / add a document to the knowledge base
2. "rag_query"     — user wants to retrieve information or get an answer from existing documents
                    NOTE: questions asking WHAT, HOW, WHY, WHEN about a topic are ALWAYS rag_query
                    even if they sound like a complaint or support issue
3. "support_query" — user has a PERSONAL problem that needs human/agent intervention
                    NOTE: support_query is only when the user is describing THEIR OWN specific problem
                    not when they are asking a general information question

Key rules:
- "What is X?" → ALWAYS rag_query
- "How do I X?" → ALWAYS rag_query (asking for instructions, not reporting a problem)
- "I want to / I need to X" → support_query ONLY if it requires account intervention
- "I have a problem with X" → ALWAYS support_query
- "I [past tense personal event]" → ALWAYS support_query (e.g. "I was charged", "I missed")

Respond ONLY with a valid JSON object in this exact format — no preamble, no explanation:
{
    "intent": "rag_query",
    "confidence": 0.95,
    "reasoning": "one short sentence"
}
"""


def classify_intent(query: str) -> dict:
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": INTENT_SYSTEM_PROMPT},
            {"role": "user", "content": query},
        ],
        temperature=0.1,
    )

    raw = response.choices[0].message.content.strip()

    try:
        result = json.loads(raw)
    except json.JSONDecodeError:
        import re

        match = re.search(r"\{.*\}", raw, re.DOTALL)
        if match:
            result = json.loads(match.group())
        else:
            raise ValueError(f"Could not parse intent response: {raw}")

    return result


def intent_node(state: AgentState) -> dict:
    query = state["query"]
    result = classify_intent(query=query)
    intent = result.get("intent", "rag_query")
    confidence = result.get("confidence", 0.0)

    valid_intents = {"ingest", "rag_query", "support_query"}
    if intent not in valid_intents:
        intent = "rag_query"
        confidence = 0.0

    print(f"✅ intent_node → intent: {intent}  confidence: {confidence}")

    return {
        "intent": intent,
        "confidence": confidence,
    }

In [19]:
state = {"user_id": "1", "query": "What is the refund policy?"}

result = intent_node(state)
print(result)

✅ intent_node → intent: rag_query  confidence: 0.95
{'intent': 'rag_query', 'confidence': 0.95}
